# Expurgo Raw: TTL de 48h

**Objetivo:** remover da tabela `raw.vendas` todos os registros ingeridos há mais de 48 horas, conforme a decisão de arquitetura de manter a camada Raw como um armazenamento temporário para dados transacionais de alto volume.

**Escopo:** aplica-se **apenas** à tabela `raw.vendas`. As tabelas de dimensão (`raw.produtos`, `raw.lojas`, `raw.representantes`) têm retenção permanente, por serem tabelas de baixo volume e carga única — não fazem parte do escopo deste expurgo.

**Lógica:** filtra pela coluna de partição `data_ingestao_particao`, removendo tudo anterior a "hoje menos 48 horas". Como a tabela é particionada por essa coluna, o `DELETE` se beneficia de **partition pruning** (remoção eficiente por partição inteira, sem varrer linha a linha).

**Complementar:** roda `VACUUM` após o delete, para remover fisicamente os arquivos órfãos do disco (o Delta Lake, por padrão, mantém arquivos antigos por um tempo, mesmo após um DELETE lógico, para viabilizar Time Travel).

In [0]:
# Calculo da data limite

from datetime import datetime, timedelta

agora = datetime.now()
data_limite = (agora - timedelta(hours=48)).date()

print(f"Agora: {agora}")
print(f"Data limite (48h atrás): {data_limite}")
print(f"Registros com data_ingestao_particao < '{data_limite}' serão removidos.")

In [0]:
# Expurgo - DELETE de registros com mais de 48h

spark.sql(f"""
    DELETE FROM poc_latam_food.raw.vendas
    WHERE data_ingestao_particao < '{data_limite}'
""")

print(f"Expurgo concluído: registros com data_ingestao_particao < '{data_limite}' foram removidos.")

In [0]:
# Validação pós-expurgo

print(f"Total de linhas após expurgo: {spark.table('poc_latam_food.raw.vendas').count()}")

print("Contagem por partição (data_ingestao_particao) após expurgo:")
spark.table("poc_latam_food.raw.vendas").groupBy("data_ingestao_particao").count().orderBy("data_ingestao_particao").display()

In [0]:
# Verificação do histórico da tabela

spark.sql("DESCRIBE HISTORY poc_latam_food.raw.vendas").display()

In [0]:
# Checagem direta

spark.sql("""
    SELECT count(*) as total_dia_17
    FROM poc_latam_food.raw.vendas
    WHERE data_ingestao_particao = '2026-07-17'
""").display()